# M4.1: chunking strategy evaluation

This notebook compares Fixed, Recursive, and the production Structure-Aware Chunker. The dataset, embedding model, query encoding, cosine retrieval, and metrics are fixed; chunking strategy is the only intended variable. Ground truth uses source block IDs, never generated chunk IDs.

In [ ]:
# Run in a fresh Google Colab runtime. Replace the repository URL if using a fork.
import os
import subprocess
from pathlib import Path

REPOSITORY_URL = 'https://github.com/your-org/prompt-generator-rag.git'  # Replace with your repository URL.
repository = Path('prompt-generator-rag')
if not repository.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL], check=True)
os.chdir(repository)
subprocess.run(['pip', 'install', '-q', '-e', 'packages/prompt-engine'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', 'apps/api', '--no-deps', 'sentence-transformers'], check=True)

In [ ]:
from pathlib import Path
from evals.src.dataset import load_dataset
from evals.src.chunking_eval import (
    SentenceTransformerEmbedder, fixed_size_chunks, recursive_chunks,
    production_structure_aware_chunks, run_comparison, save_results,
)
from app.document_processing.models import ChunkingConfig

ROOT = Path.cwd()
dataset = load_dataset(ROOT / 'evals/datasets/chunking_eval_v1.json')
MODEL = 'Alibaba-NLP/gte-multilingual-base'
embedder = SentenceTransformerEmbedder(MODEL)

In [ ]:
production_config = ChunkingConfig(target_tokens=350, max_tokens=500, overlap_tokens=40)
strategies = {
    'fixed': tuple(chunk for document in dataset.documents for chunk in fixed_size_chunks(document, max_tokens=500, overlap_tokens=40)),
    'recursive': tuple(chunk for document in dataset.documents for chunk in recursive_chunks(document, target_tokens=350, max_tokens=500)),
    # Calls apps/api/app/document_processing/chunking.py; it is not copied here.
    'production_structure_aware': tuple(chunk for document in dataset.documents for chunk in production_structure_aware_chunks(document, config=production_config)),
}
configurations = {
    'fixed': {'max_tokens': 500, 'overlap_tokens': 40},
    'recursive': {'target_tokens': 350, 'max_tokens': 500},
    'production_structure_aware': {'target_tokens': 350, 'max_tokens': 500, 'overlap_tokens': 40},
}
results = run_comparison(dataset, embedder=embedder, strategies=strategies)

In [ ]:
import pandas as pd

comparison = pd.DataFrame([{'Chunker': result.name, 'Avg Tokens': result.chunk_statistics['mean_tokens'], 'Recall@5': result.overall['recall_at_5'], 'Recall@10': result.overall['recall_at_10'], 'MRR': result.overall['mrr'], 'nDCG@10': result.overall['ndcg_at_10'], 'HitRate@5': result.overall['hit_rate_at_5']} for result in results])
display(comparison)
for metric, label in [('Recall@10', 'best Recall@10'), ('MRR', 'best MRR'), ('nDCG@10', 'best nDCG@10')]:
    winners = comparison.loc[comparison[metric] == comparison[metric].max(), 'Chunker'].tolist()
    print(f'{label}: {winners}')
for result in results:
    print(f'\n{result.name}: category breakdown')
    display(pd.DataFrame(result.by_category).T)
    print(f'{result.name}: language breakdown')
    display(pd.DataFrame(result.by_language).T)

save_results(results, dataset_version=dataset.version, embedder=embedder, output_dir=ROOT / 'evals/results/chunking', chunker_configurations=configurations)

## Reporting rule

Report the best Recall@10, MRR, and nDCG@10 independently. Recommend a strategy only when it wins the quality metric relevant to the use case without an unacceptable chunk-size or truncation tradeoff; otherwise report the tradeoff. Do not create a composite score.